In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    driver.get("http://localhost:5173/login")

    wait.until(EC.presence_of_element_located((By.ID, "username")))
    driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
    driver.find_element(By.ID, "password").send_keys("12345678")
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(3)

    # Dashboard is the default staff module; confirm it loaded (verified markers)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//section[@aria-label='Pharmacist dashboard']")))
    url_before = driver.current_url
    print("Before refresh:", url_before)

    driver.refresh()
    time.sleep(3)

    # App re-verifies the token and renders StaffApp again (verified in App.jsx)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//nav[@aria-label='Staff']")))
    wait.until(EC.visibility_of_element_located((By.XPATH, "//section[@aria-label='Pharmacist dashboard']")))
    assert not driver.find_elements(By.ID, "username"), "Session lost: login form shown after refresh."
    token = driver.execute_script("return window.localStorage.getItem('pharvo_access_token');")
    assert token, "Session lost: access token missing after refresh."
    body = driver.find_element(By.TAG_NAME, "body").text
    assert len(body.strip()) > 200, "Page blank after refresh."
    print("After refresh:", driver.current_url)
    print("PASS: Page refresh handled successfully")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("43_page_refresh_FAIL.png")
finally:
    driver.quit()